# LeetCode #1386: Cinema Seat Allocation

https://leetcode.com/problems/cinema-seat-allocation/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(r \times s)$ | $O(r \times s)$ |
| **Optimal: Per-Row Bitmask ★** | $O(r + s)$ | $O(r)$ |

---

## Understanding the Methods

### Brute Force
Allocate a full $r \times 10$ boolean grid for all rows, mark reserved seats, then scan each row for valid 4-seat blocks. Uses $O(r \times s)$ space, where $r$ = number of rows and $s$ = seats per row.

### Optimal: Per-Row Bitmask ★
Only rows that have at least one reserved seat need special handling. Represent each such row's reserved columns as a 10-bit integer. Check the three valid 4-seat blocks — columns [2-5], [4-7], [6-9] — using bitmask AND against pre-computed masks. Rows with no reservations always contribute 2 groups.

**Why this is better than Brute Force:** Space drops from $O(r \times 10)$ to $O(\text{reserved rows})$; column scanning is replaced by 3 bitmask tests per row.

**Constraints:**
* $1 \leq n \leq 10^9$ (rows), $1 \leq \text{reservedSeats.length} \leq \min(10n, 10^4)$
* $1 \leq \text{col} \leq 10$, aisle seats 1 and 10 can never be part of a valid group
* Valid 4-seat blocks: columns [2,3,4,5], [4,5,6,7], [6,7,8,9]


## Solutions

### C#

In [ ]:
public class Solution {
    public int MaxNumberOfFamilies(int n, int[][] reservedSeats) {
        // Bitmask for each row that has at least one reservation (cols 1–10 → bits 0–9)
        var rowMask = new Dictionary<int, int>();
        foreach (var seat in reservedSeats)
            rowMask[seat[0]] = rowMask.GetValueOrDefault(seat[0], 0) | (1 << (seat[1] - 1));

        // Pre-computed column masks (0-indexed bits): left=[2-5], mid=[4-7], right=[6-9]
        int LEFT  = 0b0011110;  // cols 2,3,4,5
        int MID   = 0b1111000;  // cols 4,5,6,7  (wait: bit positions 3-6)
        int RIGHT = 0b111100000; // cols 6,7,8,9

        // Correct bit layout: col c uses bit (c-1)
        // col 2=bit1, 3=bit2, 4=bit3, 5=bit4 -> LEFT  = (1<<1)|(1<<2)|(1<<3)|(1<<4) = 0b11110
        // col 4=bit3, 5=bit4, 6=bit5, 7=bit6 -> MID   = 0b1111000
        // col 6=bit5, 7=bit6, 8=bit7, 9=bit8 -> RIGHT = 0b111100000
        LEFT  = (1<<1)|(1<<2)|(1<<3)|(1<<4);
        MID   = (1<<3)|(1<<4)|(1<<5)|(1<<6);
        RIGHT = (1<<5)|(1<<6)|(1<<7)|(1<<8);

        // Rows with no reservations always fit 2 groups
        int count = 2 * (n - rowMask.Count);

        foreach (var kv in rowMask) {
            int mask = kv.Value;
            bool leftFree  = (mask & LEFT)  == 0;
            bool midFree   = (mask & MID)   == 0;
            bool rightFree = (mask & RIGHT) == 0;

            if (leftFree && rightFree) count += 2;      // both sides free
            else if (leftFree || midFree || rightFree) count += 1;
        }
        return count;
    }
}

### Python

In [ ]:
class Solution:
    def maxNumberOfFamilies(self, n: int, reservedSeats: list[list[int]]) -> int:
        # col c uses bit (c-1); valid group masks (cols 2-5, 4-7, 6-9)
        LEFT  = 0b0_0001_1110   # cols 2,3,4,5 -> bits 1,2,3,4
        MID   = 0b0_0111_1000   # cols 4,5,6,7 -> bits 3,4,5,6
        RIGHT = 0b1_1110_0000   # cols 6,7,8,9 -> bits 5,6,7,8

        # Build per-row bitmask only for rows with reservations
        row_mask: dict[int, int] = {}
        for row, col in reservedSeats:
            row_mask[row] = row_mask.get(row, 0) | (1 << (col - 1))

        # Unreserved rows always accommodate two groups
        count = 2 * (n - len(row_mask))

        for mask in row_mask.values():
            left_free  = (mask & LEFT)  == 0
            mid_free   = (mask & MID)   == 0
            right_free = (mask & RIGHT) == 0

            if left_free and right_free:
                count += 2      # both 4-seat blocks on opposite sides are free
            elif left_free or mid_free or right_free:
                count += 1      # at least one valid block remains

        return count

### Go

In [ ]:
func maxNumberOfFamilies(n int, reservedSeats [][]int) int {
    // Bit (col-1) represents seat in column col; valid block masks:
    const LEFT  = (1<<1)|(1<<2)|(1<<3)|(1<<4) // cols 2-5
    const MID   = (1<<3)|(1<<4)|(1<<5)|(1<<6) // cols 4-7
    const RIGHT = (1<<5)|(1<<6)|(1<<7)|(1<<8) // cols 6-9

    rowMask := map[int]int{}
    for _, seat := range reservedSeats {
        rowMask[seat[0]] |= 1 << (seat[1] - 1)
    }

    // Rows without any reservation always fit two groups
    count := 2 * (n - len(rowMask))

    for _, mask := range rowMask {
        leftFree  := mask&LEFT  == 0
        midFree   := mask&MID   == 0
        rightFree := mask&RIGHT == 0

        if leftFree && rightFree {
            count += 2 // both side blocks clear
        } else if leftFree || midFree || rightFree {
            count += 1 // at least one block is fully unobstructed
        }
    }
    return count
}

### Rust

In [ ]:
use std::collections::HashMap;

impl Solution {
    pub fn max_number_of_families(n: i32, reserved_seats: Vec<Vec<i32>>) -> i32 {
        // Bit positions: col c -> bit (c-1); valid group masks
        const LEFT:  i32 = (1<<1)|(1<<2)|(1<<3)|(1<<4); // cols 2-5
        const MID:   i32 = (1<<3)|(1<<4)|(1<<5)|(1<<6); // cols 4-7
        const RIGHT: i32 = (1<<5)|(1<<6)|(1<<7)|(1<<8); // cols 6-9

        let mut row_mask: HashMap<i32, i32> = HashMap::new();
        for seat in &reserved_seats {
            *row_mask.entry(seat[0]).or_insert(0) |= 1 << (seat[1] - 1);
        }

        // Every row with no reservations contributes 2 groups automatically
        let mut count: i32 = 2 * (n - row_mask.len() as i32);

        for &mask in row_mask.values() {
            let left_free  = mask & LEFT  == 0;
            let mid_free   = mask & MID   == 0;
            let right_free = mask & RIGHT == 0;

            if left_free && right_free {
                count += 2; // both side blocks unobstructed
            } else if left_free || mid_free || right_free {
                count += 1; // exactly one valid block remains
            }
        }
        count
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `n = 3`, `reservedSeats = [[1,2],[1,3],[1,9]]`
Row 1 mask: bits 1,2,8 set. LEFT (bits 1-4) has bit 1 set → not free. MID (bits 3-6) all clear → free (+1). RIGHT (bits 5-8) has bit 8 set → not free. Row 1 contributes **1**. Rows 2,3 unreserved → 2 groups each. Total = **5**.

### 2. Slightly Complex
**Input:** `n = 2`, `reservedSeats = [[2,1],[1,8]]`
Row 2: col 1 (bit 0) reserved — but col 1 is an aisle seat outside all valid blocks. LEFT/MID/RIGHT masks don't include bit 0 → all blocks free → **2**. Row 1: col 8 (bit 7) blocks RIGHT (bits 5-8) → only LEFT and MID free. LEFT free → +1. Row 1 contributes **1**. Total = **3**.

### 3. Edge Case: Time Factor
**Input:** $n = 10^9$, $10^4$ reserved seats all in distinct rows
$10^4$ rows have one reserved seat each; $10^9 - 10^4$ rows are fully free. Fully-free rows add $2(10^9 - 10^4)$ in $O(1)$ arithmetic. The reserved-row loop iterates $10^4$ times. Total $O(s)$ where $s = \text{reservedSeats.length}$.

### 4. Edge Case: Space Factor
**Input:** $n = 10^9$, all $10^4$ reserved seats in the same row
`rowMask` has exactly **1** entry. The rest of the $10^9 - 1$ rows each contribute 2 groups computed in $O(1)$. Space is $O(1)$ for the single-entry map — demonstrates how the bitmask approach scales by row count without extra memory.

### 5. Almost-Impossible but Plausible
**Input:** `n = 1`, `reservedSeats = [[1,4],[1,5]]`
Cols 4,5 (bits 3,4) block both LEFT (needs bits 1-4) and MID (needs bits 3-6). RIGHT (bits 5-8): col 5 = bit 4, not in RIGHT. Wait — col 5 = bit 4 which IS in MID (bit 4 = $2^4$, MID includes $2^3|2^4|2^5|2^6$). RIGHT = bits 5-8: col 5 = bit 4 is NOT in RIGHT. So RIGHT is free → count += 1. Total = **1**. Confirms that reservations in the middle can block left and center but leave the right block open.
